# 🎬 Netflix ETL Pipeline — Arquitectura Medallion

**Autor:** Miguel Angel Núñez Martínez  
**Dataset:** [Netflix Movies and TV Shows — Kaggle](https://www.kaggle.com/datasets/shivamb/netflix-shows)  
**Stack:** Python · Pandas · Matplotlib · Seaborn  

---

## 📐 Arquitectura del Pipeline

```
  [Kaggle CSV]
       │
       ▼
  🥉 BRONZE  →  Datos crudos + timestamp de ingesta  (Parquet)
       │
       ▼
  🥈 SILVER  →  Limpieza, validación, normalización  (Parquet)
       │
       ▼
  🥇 GOLD    →  Tablas analíticas listas para negocio (Parquet/CSV)
```


## ⚙️ 0. Instalación y configuración

In [ ]:
# Instalar dependencias necesarias
!pip install kaggle pandas pyarrow matplotlib seaborn --quiet
print('✅ Dependencias instaladas')

In [ ]:
import os
import logging
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime
from pathlib import Path

# ── Logging ──────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('netflix_etl')

# ── Estilo de gráficos ────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# ── Rutas del proyecto ────────────────────────────────────────
BASE_PATH = Path('/content/data')
BRONZE_PATH = BASE_PATH / 'bronze'
SILVER_PATH = BASE_PATH / 'silver'
GOLD_PATH   = BASE_PATH / 'gold'

for path in [BRONZE_PATH, SILVER_PATH, GOLD_PATH]:
    path.mkdir(parents=True, exist_ok=True)

logger.info('Estructura de directorios lista')
print('\n📁 Estructura creada:')
print('  data/bronze/  → datos crudos')
print('  data/silver/  → datos limpios')
print('  data/gold/    → datos analíticos')

## 📥 1. Extracción — Descarga del Dataset

> **Instrucciones:** Descarga el archivo `netflix_titles.csv` desde  
> 👉 https://www.kaggle.com/datasets/shivamb/netflix-shows  
> y súbelo a Colab con el bloque de código de abajo.

In [ ]:
# Opción A: subir manualmente desde tu computadora
from google.colab import files

print('📂 Selecciona el archivo netflix_titles.csv de tu computadora:')
uploaded = files.upload()

# Guardamos la ruta del CSV subido
CSV_PATH = list(uploaded.keys())[0]
print(f'\n✅ Archivo recibido: {CSV_PATH}')

## 🥉 2. Capa Bronze — Raw Data

Objetivo: guardar los datos **exactamente como llegan**, sin modificar nada.  
Solo agregamos un campo `ingestion_timestamp` para saber cuándo fue la carga.

In [ ]:
def extract_to_bronze(csv_path: str) -> pd.DataFrame:
    """
    Lee el CSV crudo y lo persiste en Bronze como Parquet.
    No aplica ninguna transformación a los datos originales.

    Args:
        csv_path: ruta al archivo CSV de Netflix

    Returns:
        DataFrame con los datos crudos + timestamp
    """
    logger.info('BRONZE: iniciando extracción...')

    # Leer CSV sin modificar nada
    df = pd.read_csv(csv_path, dtype=str)  # dtype=str para no perder nada
    logger.info(f'BRONZE: {len(df):,} registros leídos del CSV')

    # Único campo que agregamos: timestamp de ingesta
    df['ingestion_timestamp'] = datetime.utcnow().isoformat()

    # Guardar como Parquet (más eficiente que CSV para pipelines)
    output_path = BRONZE_PATH / 'netflix_raw.parquet'
    df.to_parquet(output_path, index=False)
    logger.info(f'BRONZE: archivo guardado en {output_path}')

    return df


# ── Ejecutar ─────────────────────────────────────────────────
df_bronze = extract_to_bronze(CSV_PATH)

print('\n🥉 BRONZE — Vista previa:')
print(f'   Filas: {len(df_bronze):,}  |  Columnas: {df_bronze.shape[1]}')
df_bronze.head(3)

In [ ]:
# Revisión inicial de calidad de datos
print('🔍 Valores nulos por columna (Bronze):')
nulls = df_bronze.isnull().sum()
nulls_pct = (nulls / len(df_bronze) * 100).round(1)
null_report = pd.DataFrame({'Nulos': nulls, '% del total': nulls_pct})
print(null_report[null_report['Nulos'] > 0].to_string())

## 🥈 3. Capa Silver — Cleaned Data

Aplicamos limpieza y validación:
- Eliminar duplicados
- Manejar valores nulos
- Corregir tipos de datos
- Normalizar formatos
- Separar el campo `listed_in` (géneros) en lista

In [ ]:
def clean_nulls(df: pd.DataFrame) -> pd.DataFrame:
    """Maneja los valores nulos según el tipo de columna."""
    df = df.copy()

    # Columnas de texto: rellenar con 'Unknown'
    text_cols = ['director', 'cast', 'country']
    for col in text_cols:
        df[col] = df[col].fillna('Unknown')

    # Rating: rellenar con el más frecuente (moda)
    if df['rating'].isnull().any():
        moda = df['rating'].mode()[0]
        df['rating'] = df['rating'].fillna(moda)
        logger.info(f'SILVER: rating nulo rellenado con moda: {moda}')

    # Eliminar filas sin fecha o sin duración (datos críticos)
    before = len(df)
    df = df.dropna(subset=['date_added', 'duration'])
    logger.info(f'SILVER: eliminadas {before - len(df)} filas sin date_added/duration')

    return df


def fix_types(df: pd.DataFrame) -> pd.DataFrame:
    """Convierte columnas a sus tipos correctos."""
    df = df.copy()

    # Fecha en formato datetime
    df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')

    # Año de lanzamiento como entero
    df['release_year'] = pd.to_numeric(df['release_year'], errors='coerce').astype('Int64')

    # Extraer año y mes de la fecha de adición
    df['year_added']  = df['date_added'].dt.year.astype('Int64')
    df['month_added'] = df['date_added'].dt.month.astype('Int64')

    logger.info('SILVER: tipos de datos corregidos')
    return df


def normalize_genres(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia y normaliza la columna de géneros."""
    df = df.copy()
    # Limpiar espacios en cada género
    df['listed_in'] = df['listed_in'].str.strip()
    # Crear lista de géneros separados por coma
    df['genres_list'] = df['listed_in'].str.split(',').apply(
        lambda x: [g.strip() for g in x] if isinstance(x, list) else []
    )
    logger.info('SILVER: columna de géneros normalizada')
    return df


def transform_to_silver(df_bronze: pd.DataFrame) -> pd.DataFrame:
    """
    Orquesta todas las transformaciones Silver.

    Args:
        df_bronze: DataFrame de la capa Bronze

    Returns:
        DataFrame limpio y validado
    """
    logger.info('SILVER: iniciando transformaciones...')
    df = df_bronze.copy()

    # 1. Eliminar duplicados
    before = len(df)
    df = df.drop_duplicates(subset=['show_id'])
    logger.info(f'SILVER: {before - len(df)} duplicados eliminados')

    # 2. Limpiar nulos
    df = clean_nulls(df)

    # 3. Corregir tipos
    df = fix_types(df)

    # 4. Normalizar géneros
    df = normalize_genres(df)

    # 5. Eliminar columna de ingesta (ya no necesaria)
    df = df.drop(columns=['ingestion_timestamp'])

    # Guardar Silver
    output_path = SILVER_PATH / 'netflix_clean.parquet'
    df.drop(columns=['genres_list']).to_parquet(output_path, index=False)  # parquet no soporta listas
    logger.info(f'SILVER: guardado en {output_path}')

    return df


# ── Ejecutar ─────────────────────────────────────────────────
df_silver = transform_to_silver(df_bronze)

print('\n🥈 SILVER — Resultado:')
print(f'   Filas: {len(df_silver):,}  |  Columnas: {df_silver.shape[1]}')
print(f'   Tipos: date_added={df_silver["date_added"].dtype}, release_year={df_silver["release_year"].dtype}')
df_silver[['title', 'type', 'date_added', 'release_year', 'rating', 'listed_in']].head(4)

## 🥇 4. Capa Gold — Business Ready

Creamos 3 tablas analíticas listas para consumo:

| Tabla | Descripción |
|---|---|
| `gold_content_per_year` | Contenido agregado por año y tipo |
| `gold_top_genres` | Géneros más frecuentes en el catálogo |
| `gold_type_distribution` | Distribución Movies vs TV Shows |

In [ ]:
def build_gold_content_per_year(df: pd.DataFrame) -> pd.DataFrame:
    """
    Tabla Gold 1: cuántos títulos se agregaron por año y tipo.
    Útil para ver el crecimiento del catálogo de Netflix.
    """
    gold = (
        df.dropna(subset=['year_added'])
          .groupby(['year_added', 'type'])
          .agg(total_titles=('show_id', 'count'))
          .reset_index()
          .sort_values('year_added')
    )
    gold['year_added'] = gold['year_added'].astype(int)
    return gold


def build_gold_top_genres(df: pd.DataFrame, top_n: int = 15) -> pd.DataFrame:
    """
    Tabla Gold 2: géneros más populares en el catálogo.
    Explota la lista de géneros para contar cada uno individualmente.
    """
    genres_exploded = df.explode('genres_list')
    genres_exploded['genres_list'] = genres_exploded['genres_list'].str.strip()

    gold = (
        genres_exploded
        .groupby('genres_list')
        .agg(total_titles=('show_id', 'count'))
        .reset_index()
        .rename(columns={'genres_list': 'genre'})
        .sort_values('total_titles', ascending=False)
        .head(top_n)
    )
    return gold


def build_gold_type_distribution(df: pd.DataFrame) -> pd.DataFrame:
    """
    Tabla Gold 3: distribución porcentual de Movies vs TV Shows.
    """
    gold = (
        df.groupby('type')
          .agg(total=('show_id', 'count'))
          .reset_index()
    )
    gold['percentage'] = (gold['total'] / gold['total'].sum() * 100).round(1)
    return gold


def aggregate_to_gold(df_silver: pd.DataFrame) -> dict:
    """
    Orquesta la creación de todas las tablas Gold y las persiste.

    Returns:
        dict con los tres DataFrames Gold
    """
    logger.info('GOLD: construyendo tablas analíticas...')

    tables = {
        'content_per_year':   build_gold_content_per_year(df_silver),
        'top_genres':         build_gold_top_genres(df_silver),
        'type_distribution':  build_gold_type_distribution(df_silver),
    }

    for name, df in tables.items():
        path = GOLD_PATH / f'gold_{name}.parquet'
        df.to_parquet(path, index=False)
        logger.info(f'GOLD: {name} guardado ({len(df)} filas) → {path}')

    return tables


# ── Ejecutar ─────────────────────────────────────────────────
gold_tables = aggregate_to_gold(df_silver)

print('\n🥇 GOLD — Tablas generadas:')
for name, df in gold_tables.items():
    print(f'   • {name}: {len(df)} filas')

In [ ]:
# Vista previa de cada tabla Gold
print('📊 content_per_year (últimos 5 años):')
display(gold_tables['content_per_year'].tail(10))

print('\n📊 top_genres (top 5):')
display(gold_tables['top_genres'].head(5))

print('\n📊 type_distribution:')
display(gold_tables['type_distribution'])

## 📊 5. Visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Netflix — Análisis del Catálogo', fontsize=16, fontweight='bold', y=1.02)

# ── Gráfico 1: Contenido por año ──────────────────────────────
ax1 = axes[0]
df_year = gold_tables['content_per_year']
for content_type, group in df_year.groupby('type'):
    ax1.plot(group['year_added'], group['total_titles'],
             marker='o', linewidth=2, label=content_type)
ax1.set_title('Títulos agregados por año', fontweight='bold')
ax1.set_xlabel('Año')
ax1.set_ylabel('Cantidad de títulos')
ax1.legend()
ax1.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# ── Gráfico 2: Top géneros ────────────────────────────────────
ax2 = axes[1]
top_g = gold_tables['top_genres'].head(10)
bars = ax2.barh(top_g['genre'], top_g['total_titles'],
                color=sns.color_palette('muted', len(top_g)))
ax2.invert_yaxis()
ax2.set_title('Top 10 géneros más frecuentes', fontweight='bold')
ax2.set_xlabel('Cantidad de títulos')
for bar, val in zip(bars, top_g['total_titles']):
    ax2.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
             str(val), va='center', fontsize=9)

# ── Gráfico 3: Movies vs TV Shows ────────────────────────────
ax3 = axes[2]
dist = gold_tables['type_distribution']
colors = ['#E50914', '#221F1F']  # Colores de Netflix
wedges, texts, autotexts = ax3.pie(
    dist['total'],
    labels=dist['type'],
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    textprops={'fontsize': 11}
)
for at in autotexts:
    at.set_color('white')
    at.set_fontweight('bold')
ax3.set_title('Movies vs TV Shows', fontweight='bold')

plt.tight_layout()
plt.savefig('/content/data/gold/netflix_dashboard.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Visualizaciones guardadas en data/gold/netflix_dashboard.png')

## ✅ 6. Reporte Final del Pipeline

In [ ]:
print('=' * 55)
print('  REPORTE FINAL — Netflix ETL Pipeline')
print('=' * 55)
print(f'\n🥉 BRONZE')
print(f'   Registros extraídos:   {len(df_bronze):>7,}')
print(f'\n🥈 SILVER')
print(f'   Registros limpios:     {len(df_silver):>7,}')
print(f'   Registros eliminados:  {len(df_bronze) - len(df_silver):>7,}')
print(f'\n🥇 GOLD (tablas generadas)')
for name, df in gold_tables.items():
    print(f'   {name:<25} {len(df):>4} filas')
print(f'\n📁 Archivos guardados en /content/data/')
print('=' * 55)